In [9]:
import os
import sys
from datetime import datetime


class TeeLogger:
    def __init__(self, log_path, stream):
        self.terminal=stream
        self.log = open(log_path, 'a', encoding = 'utf-8')

    def write(self, message):
        self.terminal.write(message)
        self.log.write(message)
        self.log.flush()

    def flush(self):
        self.terminal.flush()
        self.log.flush()
    def close(self):
        self.log.close()



log_dir = '/home/cdsw/Tony/Mlops_new/審核通過模型_MLOPS_2025/logs'
os.makedirs(log_dir,exist_ok = True)

log_filename = f"predict_main_{datetime.now().strftime('%Y%m%d_%H%M%S')}.log"

log_path = os.path.join(log_dir , log_filename)

sys.stdout = TeeLogger(log_path , sys.__stdout__)
sys.stderr = TeeLogger(log_path , sys.__stderr__)


print("logging start")
import calendar
import time

sys.path.append('/home/cdsw/Tony/Mlops_new/審核通過模型_MLOPS_2025/mlops_predict')
from choose_predict_model import main_code

sys.path.append('/home/cdsw/Tony/Mlops_new/Module')
import config
from Sql_module import get_SQL_raw_data

query="""select * from s_ianleong.mlops_id_list_double"""
model_id=get_SQL_raw_data(query)
print(model_id)
wdqdwq


def make_query_mlops_predict_status(snap_date):
    return f"""
SELECT DISTINCT product,target,population
FROM s_ianleong.mlops_ref_info_double 
where snap_date = '{str(snap_date)}'
order by  product, population
"""
year=config.ym[0:4]
month=config.ym[4:6]

end_day = calendar.monthrange(int(year),int(month))[1]
snap_date = year+'/'+month+'/'+str(end_day)

query = make_query_mlops_predict_status(snap_date)


while True:
    try:
        completed_df = get_SQL_raw_data(query)


        if len(completed_df) == 41:
            print("全執行完畢")
            break

        main_code()

    except Exception as e:
        print(f"發生錯誤:{e}，跳過")
    time.sleep(30)

KeyboardInterrupt: 

# update model new prediction label

In [ ]:
import os
import sys
import time
import warnings

warnings.filterwarnings("ignore")
sys.path.append('/home/cdsw/Tony/Mlops_new/Module')
import config
from Sql_module import execute_sql

ym = config.ym
target1 = '新戶開發與靜止戶活化'
target2 = '瞌睡客戶關懷'
target3 = '客群上送預測'
target4 = '潛在高價值客戶'

In [ ]:
all_prod_list = os.listdir('/home/cdsw/Tony/Mlops_new/審核通過模型')
if '.ipynb_checkpoints' in all_prod_list : all_prod_list.remove('.ipynb_checkpoints')
if 'mlops_predict' in all_prod_list : all_prod_list.remove('mlops_predict')
if 'mlops_retrain' in all_prod_list : all_prod_list.remove('mlops_retrain')

In [ ]:
all_prod_list

In [ ]:
def set_N_mlops_ref_info_model_valid(ym):
    return """
UPDATE s_ianleong.mlops_ref_info_double  
SET model_valid_falg ='N'
"""

In [ ]:
sql = set_N_mlops_ref_info_model_valid(ym)
execute_sql(sql)

In [ ]:
def update_mlops_ref_info_model_valid(ym,string,product,target,population):
    return f"""
UPDATE s_ianleong.mlops_ref_info_double    
SET model_valid_falg ='{string}'
where product = '{product}' 
and target = '{target}' 
and population = '{population}' 
and edition = (select max(edition) from (select a.*,regexp_substr(edition,'[^.]+',1,1)*1000+regexp_substr(edition,'[^.]+',1,2)rk from s_ianleong.mlops_ref_info_double a where product = '{product}' and target = '{target}' and population = '{population}' and edition is not null order by rk desc fetch next 1 rows only))
"""

In [ ]:
for prod in all_prod_list:
    if prod == '流失預警':
        population = '不分潛客'
        sql = update_mlops_ref_info_model_valid(ym,'Y',prod,target2,population)
        execute_sql(sql)
        time.sleep(1)
    elif prod == '客群上送':
        population = '不分潛客'
        sql = update_mlops_ref_info_model_valid(ym,'Y',prod,target3,population)
        execute_sql(sql)
        time.sleep(1)
    elif prod == '潛在高價值客戶':
        population = '不分潛客'
        sql = update_mlops_ref_info_model_valid(ym,'Y',prod,target4,population)
        execute_sql(sql)
        time.sleep(1)
    else:
        population = '非潛客'
        sql = update_mlops_ref_info_model_valid(ym,'Y',prod,target1,population)
        execute_sql(sql)
        population = '潛客'
        sql = update_mlops_ref_info_model_valid(ym,'Y',prod,target1,population)
        execute_sql(sql)
        time.sleep(1)